In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "1"     # agama
os.environ["MKL_NUM_THREADS"] = "1"     # numpy, scipy
os.environ["OPENBLAS_NUM_THREADS"] = "1"    # numpy
os.environ["NUMEXPR_NUM_THREADS"] = "1"     # pandas

import agama
import torch 
import numpy as np
from scipy import integrate
from astropy import units as u

from sbi.utils import BoxUniform
from sbi.inference import SNLE, simulate_for_sbi, prepare_for_sbi
from sbi.utils import likelihood_nn

from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import pickle
import matplotlib.pyplot as plt
from galaxy_generation import generate_galaxy_multiple
from prior_generation import generate_prior
from standardization import standardize

import corner

torch.set_num_threads(4)


In [4]:
# set agama unit to be in Msun, kpc, km/s
agama.setUnits(mass=1 * u.Msun, length=1*u.kpc, velocity=1 * u.km /u.s)

In [5]:
agama.setRandomSeed(13)
torch.manual_seed(13)
np.random.seed(13)


In [51]:
with open('./inference(model 6, v3test2).pkl', 'rb') as file:
    # Load the object from the file
    inference = pickle.load(file)

In [52]:
theta_train = np.array(pd.read_csv("training_theta(poisson).csv", header=None))
x_train = np.array(pd.read_csv("training_x(poisson).csv", header=None))

t, x = standardize(theta_train, x_train)

In [53]:
# testing dataset
test_theta_raw = np.array(pd.read_csv("test_theta.csv", header=None))
test_x_raw = np.array(pd.read_csv("test_x.csv", header=None))

test_theta, index = np.unique(test_theta_raw, axis=0, return_index=True)
index = np.sort(index)
test_x = np.split(test_x_raw, index, axis=0)[1:]
test_theta = test_theta_raw[index]

In [54]:
posterior = inference.build_posterior( 
                                    mcmc_method="slice_np_vectorized", 
                                    mcmc_parameters={"warmup_steps":500,
                                                        "num_chains":32,
                                                        "num_workers": 1,
                                                        "init_strategy": "sir",
                                                        "thin": 4})

In [56]:
x_o = torch.from_numpy (test_x[0]).float()
x_o = (x_o - x[1]) / x[2]

In [57]:

s = posterior.sample((1000,), x=x_o)
# samples = s.numpy() * t[2] + t[1]

/home/tingli/anaconda3/envs/myenv/lib/python3.9/site-packages/sbi/utils/sbiutils.py:316: UserWarning: An x with a batch size of 104 was passed. It will be interpreted as a batch of independent and identically
            distributed data X={x_1, ..., x_n}, i.e., data generated based on the
            same underlying (unknown) parameter. The resulting posterior will be with
            respect to entire batch, i.e,. p(theta | X).
  warnings.warn(
/home/tingli/anaconda3/envs/myenv/lib/python3.9/site-packages/sbi/inference/posteriors/mcmc_posterior.py:346: UserWarning: As of sbi v0.19.0, the behavior of the SIR initialization for MCMC has changed. If you wish to restore the behavior of sbi v0.18.0, set `init_strategy='resample'.`
  warn(


Running vectorized MCMC with 32 chains:   0%|          | 0/68000 [00:00<?, ?it/s]

In [58]:
s.median(dim=0)

torch.return_types.median(
values=tensor([ 6.7276, -0.3080,  0.6470,  0.5289]),
indices=tensor([ 61, 765, 941, 230]))

In [50]:
(s * t[2] + t[1]).median(dim=0)

torch.return_types.median(
values=tensor([ 7.1593, -0.4877,  0.2272,  0.8143], dtype=torch.float64),
indices=tensor([914, 409, 989, 913]))

In [31]:
test_theta[0]

array([ 6.727564  , -0.35557473,  0.8577411 ,  0.5492028 ])

In [37]:
x_o

tensor([[-1.0665e-01,  9.2206e-02, -1.9836e-02, -1.3463e-01, -4.8329e-02,
          8.8707e-02],
        [-2.0789e-02,  1.2631e-01, -1.1340e-01,  1.3773e-02,  3.5256e-02,
          1.5380e-01],
        [-1.3957e-03, -2.0314e-03, -3.4692e-02,  8.1317e-02, -1.9552e-01,
         -2.6181e-01],
        [-3.1022e-03, -5.1056e-02, -1.9342e-02, -2.5462e-01, -2.1556e-02,
          2.1418e-01],
        [-1.0442e-01,  3.0220e-01,  4.5459e-02,  2.3096e-01, -4.1039e-02,
          2.3863e-01],
        [ 1.2753e-02,  1.1247e-01,  2.7781e-01,  9.1302e-03,  6.9325e-02,
         -1.4216e-01],
        [ 1.3891e-02,  1.2019e-01,  8.2573e-02,  8.3598e-02, -1.5687e-01,
          1.4751e-01],
        [-4.0674e-02, -3.8906e-02,  5.2441e-02, -1.4447e-01, -3.0561e-02,
          1.7785e-02],
        [-6.9894e-03, -4.3151e-02, -2.1184e-02, -8.6855e-02, -1.4791e-02,
         -1.5605e-01],
        [ 5.4035e-02,  3.9162e-02, -1.0430e-01,  5.9435e-03,  2.2634e-02,
         -6.5650e-03],
        [ 5.6229e-03, -2.4745e

In [38]:
with open('./inference(model 6, v3test2).pkl', 'rb') as file:
    # Load the object from the file
    inference2 = pickle.load(file)

In [40]:
inference2.summary

{'epochs_trained': [36],
 'best_validation_log_prob': [2.885399648823689],
 'validation_log_probs': [2.6365660588765882,
  2.6975131394322385,
  2.7822604093355,
  2.782215879135525,
  2.8089245938763177,
  2.8432142783686056,
  2.8405558522214593,
  2.8468838352517984,
  2.8498334319321152,
  2.8485947687601305,
  2.8583450993311774,
  2.8656040663571702,
  2.866417732435403,
  2.8704298796113004,
  2.8586424576867486,
  2.861059351065724,
  2.8657579299100897,
  2.8820623909075236,
  2.8799729973999497,
  2.8680672842202726,
  2.871688975501306,
  2.8750168810185697,
  2.8684502926069437,
  2.8774718058477973,
  2.8850036896381184,
  2.885399648823689,
  2.868540304223287,
  2.8701156144289626,
  2.8639943894651747,
  2.876762892782074,
  2.8689158773913825,
  2.8772932524533616,
  2.8844657276094576,
  2.881656840904472,
  2.8740981215054227,
  2.879605735700155],
 'training_log_probs': [1.9106641204924102,
  2.7569468385380325,
  2.8277098847030917,
  2.860313251530001,
  2.8730336

In [6]:
with open("tune(model 6, v4).pkl", "rb") as file:
    r = pickle.load(file)

In [9]:
r.best_trial

FrozenTrial(number=23, state=<TrialState.COMPLETE: 1>, values=[2.8989651111456065], datetime_start=datetime.datetime(2026, 6, 2, 15, 15, 2, 924165), datetime_complete=datetime.datetime(2026, 6, 2, 15, 18, 45, 339205), params={'learning_rate': 0.0015121146679622883, 'training_batch_size': 512}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.01, log=True, low=0.0001, step=None), 'training_batch_size': CategoricalDistribution(choices=(256, 512, 1024, 2048))}, trial_id=23, value=None)

In [10]:
with open("samples(model 6, v4).pkl", "rb") as file:
    s = pickle.load(file)

In [11]:
s

[array([[10.84362965, -0.64033412, -0.35222583,  0.65746976],
        [10.84389858, -0.63790071, -0.35924626,  0.75434272],
        [10.84920278, -0.64031926, -0.35746381,  0.68794505],
        ...,
        [10.84301661, -0.63916378, -0.35874853,  0.70533536],
        [10.84873226, -0.63922882, -0.35774584,  0.66886434],
        [10.84579677, -0.63464714, -0.36071529,  0.66855253]]),
 array([[10.85134511, -0.64008554, -0.35394025,  0.64430752],
        [10.84635859, -0.63789844, -0.35590036,  0.64439994],
        [10.85493908, -0.63959416, -0.35755622,  0.64484292],
        ...,
        [10.85068622, -0.63703909, -0.35550411,  0.64450425],
        [10.84413115, -0.63771634, -0.35728558,  0.64454785],
        [10.84927094, -0.63947516, -0.35770353,  0.65090736]]),
 array([[10.85579254, -0.64002687, -0.35625161,  0.76407142],
        [10.87194349, -0.6394317 , -0.35184974,  0.80219573],
        [10.8514992 , -0.63369902, -0.35901726,  0.81423917],
        ...,
        [10.84688611, -0.63